In [1]:
import re
import numpy as np


def simple_tokenize(text: str):
    """
    - Word2Vec은 "단어(토큰) 시퀀스"가 입력이어야 학습이 가능함
    - 따라서 문장(string)을 토큰 리스트(list[str])로 바꿔주는 전처리가 필요함

    여기서는:
    1) 소문자화 (영어 기준 대소문자 통일)
    2) 정규식으로 "영문/숫자/한글/공백"만 남김 (기호 제거)
    3) 공백 기준 split
    """
    text = text.lower()
    # ^ ... : 해당 문자 집합이 "아닌 것"을 의미
    # 즉, 아래는 영문/숫자/한글/공백이 아닌 문자는 전부 공백으로 바꾼다.
    text = re.sub(r"[^0-9a-zA-Z가-힣\s]+", " ", text)
    return [t for t in text.split() if t]  # 빈 토큰 제거


class NumpyWord2Vec:
    """
    Minimal Word2Vec (SGNS: Skip-gram with Negative Sampling)

    핵심 아이디어(Word2Vec/SGNS):
    - 중심 단어(center word)가 주어졌을 때, 주변 단어(context word)를 맞추도록 학습
    - 실제 데이터의 (center, context) 쌍은 "positive(정답)"로 보고
    - 랜덤하게 뽑은 다른 단어들은 "negative(오답)"로 보고
    - '정답은 점수를 높이고, 오답은 점수를 낮추는' 방향으로 임베딩 벡터를 업데이트

    이 구현의 특징:
    - 외부 라이브러리(gensim) 없이 numpy만 사용
    - 학습용/소규모 예제에 적합 (성능/최적화는 최소)
    """

    def __init__(
        self,
        vector_size=50,  # 임베딩 차원 D
        window=3,        # 컨텍스트 윈도우 크기 (좌/우로 몇 칸까지 볼지)
        min_count=1,     # 이 빈도 미만 단어는 vocab에서 제거
        negative=5,      # negative sampling 개수 (positive 1개당 negative 몇 개?)
        lr=0.05,         # 학습률(gradient update step size)
        epochs=50,       # 전체 데이터 반복 횟수
        seed=42,         # 난수 시드 (재현 가능)
    ):
        # 하이퍼파라미터를 타입 강제 변환해두면
        # 실수/문자열이 들어오는 경우도 안정적으로 처리할 수 있음
        self.vector_size = int(vector_size)
        self.window = int(window)
        self.min_count = int(min_count)
        self.negative = int(negative)
        self.lr = float(lr)
        self.epochs = int(epochs)

        # numpy random generator (재현 가능한 난수)
        self.rng = np.random.default_rng(seed)

        # 단어 <-> id 매핑
        # - 행렬로 학습하려면 단어를 "정수 인덱스(id)"로 바꿔야 함
        self.word2id = {}
        self.id2word = {}

        # vocab 내 단어들의 등장 횟수(counts) 및 negative sampling 확률 분포
        self.counts = None
        self.unigram_probs = None

        # 학습 파라미터(임베딩 행렬)
        # - W_in : center word 임베딩
        # - W_out: context word 임베딩
        # 전통적인 word2vec 구현에서도 in/out 두 행렬을 따로 둠
        self.W_in = None   # shape: (V, D)
        self.W_out = None  # shape: (V, D)

    @staticmethod
    def _sigmoid(x):
        """
        시그모이드 함수 σ(x) = 1 / (1 + exp(-x))

        Word2Vec(SGNS)는 "점수(score)=내적(dot)"을 확률처럼 해석하기 위해 시그모이드를 씀.
        예: P(positive | center, context) ≈ σ(v_c · u_o)

        overflow 방지:
        - x가 매우 크거나 작으면 exp가 overflow/underflow를 일으킬 수 있음
        - clip으로 안전 범위를 제한
        """
        x = np.clip(x, -15, 15)
        return 1.0 / (1.0 + np.exp(-x))

    def _build_vocab(self, tokenized_sentences):
        """
        단어 빈도를 세고(min_count 적용),
        word2id / id2word / counts / negative sampling 분포를 만든다.

        tokenized_sentences: List[List[str]]
        """
        # 1) 단어 빈도 세기
        counts = {}
        for sent in tokenized_sentences:
            for w in sent:
                counts[w] = counts.get(w, 0) + 1

        # 2) min_count 미만 단어 제거
        vocab = [w for w, c in counts.items() if c >= self.min_count]

        # 3) 정렬(재현성과 일관성을 위해)
        vocab.sort()

        # 4) 단어 <-> id 매핑 생성
        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}

        # 5) counts를 vocab 순서대로 numpy 배열로 저장
        self.counts = np.array([counts[w] for w in vocab], dtype=np.float64)

        # 6) Negative sampling 분포 만들기
        # word2vec 논문에서 자주 쓰는 분포: unigram(w)^0.75
        # - 너무 자주 나오는 단어가 negative에 과도하게 뽑히는 걸 완화하면서도
        #   희귀 단어가 지나치게 안 뽑히는 문제를 줄이는 경험적 트릭
        p = np.power(self.counts, 0.75)
        p = p / (p.sum() if p.sum() > 0 else 1.0)
        self.unigram_probs = p

    def _init_weights(self):
        """
        임베딩 행렬 초기화.

        V: vocabulary 크기
        D: 임베딩 차원
        """
        V = len(self.word2id)
        D = self.vector_size

        # W_in은 작은 랜덤값으로 시작 (학습 시작점)
        # (rng.random - 0.5)로 [-0.5, 0.5) 근처 분포로 만든 뒤
        # D로 나눠 스케일을 줄여 초기 내적값이 너무 커지지 않게 함
        self.W_in = (self.rng.random((V, D)) - 0.5) / D

        # W_out은 0으로 시작해도 학습이 진행되며 값이 생김
        self.W_out = np.zeros((V, D), dtype=np.float64)

    def _sample_negatives(self, k, forbidden_id):
        """
        negative sampling: 오답 단어 id를 k개 뽑는다.

        forbidden_id:
        - 실제 정답(context_id)이 negative로 뽑히면 학습 신호가 꼬이므로
          최대한 피한다.

        구현 방식:
        - unigram_probs 분포에서 랜덤 샘플링
        - forbidden_id가 나오면 버리고 다시 뽑는 rejection sampling
        """
        V = len(self.word2id)
        if V <= 1:
            # vocab이 1개뿐이면 negative를 뽑을 수 없음
            return np.array([], dtype=np.int64)

        negs = []
        while len(negs) < k:
            candidate = int(self.rng.choice(V, p=self.unigram_probs))
            if candidate != forbidden_id:
                negs.append(candidate)

        return np.array(negs, dtype=np.int64)

    def _train_pair(self, center_id, context_id):
        """
        (center_id, context_id) 한 쌍에 대해 SGNS 업데이트를 수행한다.

        용어:
        - v_c: center 단어의 "입력(in)" 벡터 (W_in에서 가져옴)
        - u_o: context 단어의 "출력(out)" 벡터 (W_out에서 가져옴)

        목표:
        - positive 쌍(center, context): σ(v_c·u_o)가 1에 가까워지게(점수 ↑)
        - negative 쌍(center, neg):    σ(v_c·u_k)가 0에 가까워지게(점수 ↓)

        여기서는 완전한 벡터화 최적화 대신,
        개념이 보이도록 pair 단위로 업데이트한다.
        """
        # center 단어 벡터 (shape: (D,))
        v_c = self.W_in[center_id]

        # -------------------------
        # 1) Positive 업데이트
        # -------------------------
        # context(out) 벡터
        u_o = self.W_out[context_id]

        # 점수(score) = 내적(dot)
        score_pos = float(np.dot(v_c, u_o))

        # positive일 확률처럼 해석되는 값
        p_pos = self._sigmoid(score_pos)

        # label=1인 logistic regression의 gradient 계수 (1 - σ(score))
        # - score를 키우고 싶은 방향(정답)이므로 (1 - p_pos)
        grad_pos = (1.0 - p_pos)

        # 파라미터 업데이트:
        # - W_out[context]는 v_c 방향으로 이동 (점수 증가)
        # - W_in[center]는 u_o 방향으로 이동 (점수 증가)
        self.W_out[context_id] += self.lr * grad_pos * v_c
        self.W_in[center_id] += self.lr * grad_pos * u_o

        # -------------------------
        # 2) Negative 업데이트
        # -------------------------
        # context_id(정답)을 제외한 단어들을 negative로 k개 뽑음
        neg_ids = self._sample_negatives(self.negative, forbidden_id=context_id)

        for neg_id in neg_ids:
            u_k = self.W_out[neg_id]  # negative(out) 벡터

            score_neg = float(np.dot(v_c, u_k))
            p_neg = self._sigmoid(score_neg)

            # label=0인 logistic regression gradient 계수 (0 - σ(score)) = -σ(score)
            # - score를 낮추고 싶은 방향(오답)이므로 음수 방향 업데이트
            grad_neg = (0.0 - p_neg)

            # negative word의 out 벡터는 v_c에서 멀어지는 방향으로(점수 감소)
            # center의 in 벡터도 u_k에서 멀어지는 방향으로(점수 감소)
            self.W_out[neg_id] += self.lr * grad_neg * v_c
            self.W_in[center_id] += self.lr * grad_neg * u_k

    def fit(self, sentences):
        """
        모델 학습(훈련) 함수.

        sentences 입력 형태:
        - List[str]        : 문장 문자열 리스트 → 내부에서 토큰화 수행
        - List[List[str]]  : 이미 토큰화된 문장 리스트 → 그대로 사용

        학습 과정 요약:
        1) 토큰화 (필요 시)
        2) vocab 구축 + negative sampling 분포 구성
        3) 가중치 초기화
        4) 각 문장에 대해 window 기반 (center, context) 쌍 생성
        5) 각 쌍에 대해 SGNS 업데이트 수행
        """
        if len(sentences) == 0:
            raise ValueError("sentences is empty")

        # 1) 토큰화 여부 판단
        if isinstance(sentences[0], str):
            tokenized = [simple_tokenize(s) for s in sentences]
        else:
            tokenized = sentences

        # 2) vocab/분포 만들고 가중치 초기화
        self._build_vocab(tokenized)
        self._init_weights()

        # 3) 학습을 위해 문장을 id 시퀀스로 변환
        # - vocab에 없는 토큰은 제거
        # - 길이가 2 미만이면 (center-context) 쌍을 만들기 어려우므로 제외
        id_sents = []
        for sent in tokenized:
            ids = [self.word2id[w] for w in sent if w in self.word2id]
            if len(ids) >= 2:
                id_sents.append(ids)

        if not id_sents:
            raise ValueError("No valid training sentences after vocab/min_count filtering.")

        # 4) epochs 만큼 반복 학습
        for epoch in range(self.epochs):
            # epoch마다 문장 순서를 섞어 학습 편향을 줄임
            self.rng.shuffle(id_sents)

            # 각 문장에 대해 window 내 (center, context) 쌍을 생성해 학습
            for ids in id_sents:
                n = len(ids)
                for i, center_id in enumerate(ids):
                    # center 주변 window 범위 계산
                    left = max(0, i - self.window)
                    right = min(n, i + self.window + 1)

                    # window 범위 내의 단어들을 context로 사용
                    for j in range(left, right):
                        if j == i:
                            continue  # 자기 자신은 context로 쓰지 않음
                        context_id = ids[j]
                        self._train_pair(center_id, context_id)

        return self

    def get_vector(self, word):
        """
        단어의 임베딩 벡터를 반환한다.

        여기서는 W_in(입력 임베딩)을 사용한다.
        (구현/사용 목적에 따라 W_in, W_out, 또는 (W_in+W_out)를 쓰기도 함)

        반환 시 정규화(normalize)를 해두면
        - cosine similarity 계산이 간단해지고
        - most_similar 결과가 안정적
        """
        if word not in self.word2id:
            raise KeyError(f"'{word}' not in vocabulary")

        v = self.W_in[self.word2id[word]]

        # L2 정규화: v / ||v||
        norm = np.linalg.norm(v) + 1e-12
        return v / norm

    def most_similar(self, word, topn=5):
        """
        입력 단어와 cosine similarity가 큰 단어 topn개를 반환.

        구현:
        - 모든 단어 벡터를 정규화(Wn)
        - sims = Wn @ v  (정규화되어 있으면 내적이 cosine similarity)
        - 내림차순 정렬 후 상위 결과 반환
        """
        v = self.get_vector(word)  # 이미 정규화된 벡터
        W = self.W_in

        # 모든 vocab 벡터 정규화
        norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-12
        Wn = W / norms

        # cosine similarity 벡터 (shape: (V,))
        sims = Wn @ v

        # similarity 내림차순 정렬 인덱스
        best = np.argsort(-sims)

        results = []
        for idx in best:
            w = self.id2word[int(idx)]
            if w == word:
                continue  # 자기 자신 제외
            results.append((w, float(sims[idx])))
            if len(results) >= topn:
                break

        return results


# -----------------------------
# 예시 실행: gensim Word2Vec 사용 흐름과 비슷하게
# -----------------------------
sentences = [
    "I love machine learning and artificial intelligence",
    "Deep learning is a subset of machine learning",
    "Word embeddings are vector representations of words",
    "Natural language processing is fun",
    "Large language models are powerful",
]

# vector_size=10: 임베딩 10차원
# window=3: 주변 3단어까지 context로 사용
# negative=5: positive 1개당 negative 5개 샘플링
# epochs=200: 데이터가 매우 작으니 epoch를 늘려 학습이 좀 되게 함
w2v = NumpyWord2Vec(
    vector_size=10,
    window=3,
    min_count=1,
    negative=5,
    lr=0.05,
    epochs=200,
    seed=42,
)
w2v.fit(sentences)

word = "learning"
vector = w2v.get_vector(word)
print(f"'{word}'의 임베딩 벡터 (10차원):\n{vector}\n")

similar_words = w2v.most_similar(word, topn=3)
print(f"'{word}'와 가장 유사한 단어들: {similar_words}")


SyntaxError: invalid syntax (1462360138.py, line 378)